# Laboratorio Semana 5: Mecanismo de Atención desde cero

**Curso:** Deep Learning  
**Valor:** 4% de la nota del curso  
**Entrega:** Notebook ejecutado (.ipynb) con todas las celdas con salida visible

---

## Contexto

Esta semana extiende el Seq2Seq de la Semana 4 con el mecanismo de atencion de producto punto escalado. En lugar de un vector de contexto fijo $\mathbf{c} = \mathbf{h}_T^{enc}$, el decoder calcula en cada paso un vector de contexto **dinamico** $\tilde{\mathbf{c}}_s$ que es una combinacion ponderada de todos los hidden states del encoder.

Pipeline del modelo con atencion:
```
oracion_EN -> [E_enc] -> [Encoder LSTM] -> H_enc  (T, d_hid)
                                            |
                          K = H_enc W_K^T  |  V = H_enc W_V^T
                                            |
<SOS>+ES -> [E_dec] -> [Decoder LSTM] -> h_s^dec -> q_s = W_Q h_s^dec
                                            |
                          scores = K q_s / sqrt(d_k)
                          alpha  = softmax(scores)
                          c_tilde = V^T alpha
                                            |
                          [h_s^dec ; c_tilde] -> W_out -> prediccion
```

## Reglas

- Use unicamente PyTorch. No use `nn.MultiheadAttention` ni ninguna capa de alto nivel.
- El backward debe ser **manual**. No use `loss.backward()`.
- No modifique los pesos iniciales ni el corpus.
- La celda final calcula su nota automatica sobre los **60 puntos** de codigo.

In [1]:
import torch
import torch.nn.functional as F
import hashlib, numpy as np, random, time
import matplotlib.pyplot as plt

def _hash_tensor(t, decimals=5):
    arr = np.round(t.detach().numpy().astype(np.float64), decimals)
    return hashlib.sha256(arr.tobytes()).hexdigest()

_resultados = {}
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.13.0+cpu


---
## Bloque 0: Corpus, vocabularios y pesos (dado, no modificar)

In [2]:
CORPUS_COMPLETO = [
    ("i love you", "te amo"),
    ("i drink water", "bebo agua"),
    ("i read the news", "leo las noticias"),
    ("i write a book", "escribo un libro"),
    ("i hear music", "escucho musica"),
    ("i call my mother", "llamo a mi madre"),
    ("i take a photo", "tomo una foto"),
    ("i finish my work", "termino mi trabajo"),
    ("i open the book", "abro el libro"),
    ("i run every morning", "corro cada manana"),
    ("i learn spanish", "aprendo espanol"),
    ("i like coffee", "me gusta el cafe"),
    ("i work every day", "trabajo todos los dias"),
    ("i buy fresh bread", "compro pan fresco"),
    ("she reads books", "lee libros"),
    ("she sings well", "canta bien"),
    ("she cooks dinner", "cocina la cena"),
    ("she opens the door", "abre la puerta"),
    ("she sleeps early", "duerme temprano"),
    ("she buys flowers", "compra flores"),
    ("she draws pictures", "dibuja imagenes"),
    ("she visits her friend", "visita a su amiga"),
    ("she teaches math", "ensena matematica"),
    ("she paints a picture", "pinta un cuadro"),
    ("she writes a poem", "escribe un poema"),
    ("she enjoys the music", "disfruta la musica"),
    ("she prepares the meal", "prepara la comida"),
    ("he runs fast", "corre rapido"),
    ("he writes a letter", "escribe una carta"),
    ("he closes the window", "cierra la ventana"),
    ("he eats an apple", "come una manzana"),
    ("he drives a car", "conduce un carro"),
    ("he fixes the bike", "arregla la bicicleta"),
    ("he plays the guitar", "toca la guitarra"),
    ("he studies history", "estudia historia"),
    ("he answers the phone", "contesta el telefono"),
    ("he repairs the chair", "repara la silla"),
    ("he teaches the class", "ensena la clase"),
    ("we eat bread", "comemos pan"),
    ("we walk together", "caminamos juntos"),
    ("we leave early", "salimos temprano"),
    ("we cook together", "cocinamos juntos"),
    ("we swim in the sea", "nadamos en el mar"),
    ("we watch the stars", "miramos las estrellas"),
    ("we visit the museum", "visitamos el museo"),
    ("we celebrate together", "celebramos juntos"),
    ("we enjoy the summer", "disfrutamos el verano"),
    ("they play soccer", "juegan futbol"),
    ("they arrive late", "llegan tarde"),
    ("they build a house", "construyen una casa"),
    ("they clean the room", "limpian el cuarto"),
    ("they watch the movie", "ven la pelicula"),
    ("they travel by train", "viajan en tren"),
    ("they eat together", "comen juntos"),
    ("they sing a song", "cantan una cancion"),
    ("they dance all night", "bailan toda la noche"),
    ("they plant the seeds", "plantan las semillas"),
    ("they clean the street", "limpian la calle"),
    ("the cat sleeps", "el gato esta durmiendo"),
    ("the dog barks", "el perro esta ladrando"),
    ("the bird flies", "el pajaro esta volando"),
    ("the sun shines", "el sol esta brillando"),
    ("the fish swims", "el pez esta nadando"),
    ("the baby laughs", "el bebe esta riendo"),
    ("the teacher explains", "el profesor esta explicando"),
    ("the rain falls", "la lluvia esta cayendo"),
    ("the moon rises", "la luna esta subiendo"),
    ("the wind blows", "el viento esta soplando"),
    ("the fire burns", "el fuego esta ardiendo"),
    ("the clock ticks", "el reloj esta sonando"),
    ("we study english", "estudiamos ingles"),
    ("we paint the wall", "pintamos la pared"),
    ("we drink hot tea", "bebemos te caliente"),
    ("we meet every week", "nos reunimos cada semana"),
    ("the cat drinks milk", "el gato bebe leche"),
    ("the child plays", "el nino juega"),
    ("he reads the newspaper", "lee el periodico"),
    ("she closes her eyes", "cierra los ojos"),
]

random.seed(42)
indices = list(range(len(CORPUS_COMPLETO)))
random.shuffle(indices)
n_train = int(len(CORPUS_COMPLETO) * 0.75)
TRAIN_DATA = [CORPUS_COMPLETO[i] for i in indices[:n_train]]
TEST_DATA  = [CORPUS_COMPLETO[i] for i in indices[n_train:]]
print(f'Train: {len(TRAIN_DATA)} pares, Test: {len(TEST_DATA)} pares')
print(f'Par autograder (TRAIN_DATA[0]): {TRAIN_DATA[0]}')


Train: 58 pares, Test: 20 pares
Par autograder (TRAIN_DATA[0]): ('they play soccer', 'juegan futbol')


In [3]:
SOS, EOS, PAD, UNK = '<SOS>', '<EOS>', '<PAD>', '<UNK>'
SPECIAL = [PAD, UNK, SOS, EOS]

def build_vocab(sentences):
    words = set()
    for s in sentences: words.update(s.lower().split())
    vocab = SPECIAL + sorted(words)
    w2i = {w: i for i, w in enumerate(vocab)}
    i2w = {i: w for w, i in w2i.items()}
    return vocab, w2i, i2w

src_vocab, src_w2i, src_i2w = build_vocab([p[0] for p in CORPUS_COMPLETO])
tgt_vocab, tgt_w2i, tgt_i2w = build_vocab([p[1] for p in CORPUS_COMPLETO])
src_V = len(src_vocab); tgt_V = len(tgt_vocab)
print(f'Vocabulario EN: {src_V}, ES: {tgt_V}')

def tokenize_src(s): return [src_w2i.get(w.lower(), src_w2i[UNK]) for w in s.split()]
def tokenize_tgt(s): return ([tgt_w2i[SOS]] +
                              [tgt_w2i.get(w.lower(), tgt_w2i[UNK]) for w in s.split()] +
                              [tgt_w2i[EOS]])

Vocabulario EN: 158, ES: 163


In [4]:
# Dimensiones
d_emb = 16   # dimension de embeddings
d_hid = 32   # dimension del hidden state LSTM
d_k   = 16   # dimension del espacio de queries y keys
d_v   = 16   # dimension del espacio de values
alpha_lr = 0.01  # tasa de aprendizaje

# Pesos seq2seq base
torch.manual_seed(42)
E_enc = torch.randn(d_emb, src_V) * 0.1
E_dec = torch.randn(d_emb, tgt_V) * 0.1
Wf_enc=torch.randn(d_hid,d_hid+d_emb)*0.1; bf_enc=torch.zeros(d_hid)
Wi_enc=torch.randn(d_hid,d_hid+d_emb)*0.1; bi_enc=torch.zeros(d_hid)
Wc_enc=torch.randn(d_hid,d_hid+d_emb)*0.1; bc_enc=torch.zeros(d_hid)
Wo_enc=torch.randn(d_hid,d_hid+d_emb)*0.1; bo_enc=torch.zeros(d_hid)
Wf_dec=torch.randn(d_hid,d_hid+d_emb)*0.1; bf_dec=torch.zeros(d_hid)
Wi_dec=torch.randn(d_hid,d_hid+d_emb)*0.1; bi_dec=torch.zeros(d_hid)
Wc_dec=torch.randn(d_hid,d_hid+d_emb)*0.1; bc_dec=torch.zeros(d_hid)
Wo_dec=torch.randn(d_hid,d_hid+d_emb)*0.1; bo_dec=torch.zeros(d_hid)
# NOTA: W_out ahora recibe [h_dec ; c_tilde] de dimension (d_hid + d_v,)
W_out = torch.randn(tgt_V, d_hid + d_v) * 0.1
b_out = torch.zeros(tgt_V)

# Matrices de atencion (nuevas esta semana)
torch.manual_seed(7)
W_Q = torch.randn(d_k, d_hid) * 0.1   # (d_k, d_hid)
W_K = torch.randn(d_k, d_hid) * 0.1   # (d_k, d_hid)
W_V = torch.randn(d_v, d_hid) * 0.1   # (d_v, d_hid)

print(f'W_Q: {W_Q.shape}, W_K: {W_K.shape}, W_V: {W_V.shape}')
print(f'W_out: {W_out.shape}  <- recibe [h_dec; c_tilde] de dim {d_hid+d_v}')

W_Q: torch.Size([16, 32]), W_K: torch.Size([16, 32]), W_V: torch.Size([16, 32])
W_out: torch.Size([163, 48])  <- recibe [h_dec; c_tilde] de dim 48


In [5]:
def lstm_cell(h, c, x, Wf, bf, Wi, bi, Wc, bc, Wo, bo):
    concat = torch.cat([h, x])
    f = torch.sigmoid(Wf @ concat + bf)
    i = torch.sigmoid(Wi @ concat + bi)
    ct = torch.tanh(Wc @ concat + bc)
    cn = f * c + i * ct
    o = torch.sigmoid(Wo @ concat + bo)
    hn = o * torch.tanh(cn)
    return hn, cn, {'concat': concat, 'h_prev': h, 'c_prev': c,
                    'f': f, 'i': i, 'ct': ct, 'c_t': cn, 'o': o, 'h_t': hn}

# Forward encoder sobre el par autograder
src_test = tokenize_src(TRAIN_DATA[0][0])
h = torch.zeros(d_hid); c = torch.zeros(d_hid); enc_caches = []
for idx in src_test:
    emb = E_enc[:, idx]
    h, c, cache = lstm_cell(h, c, emb, Wf_enc, bf_enc, Wi_enc, bi_enc,
                             Wc_enc, bc_enc, Wo_enc, bo_enc)
    cache['emb_idx'] = idx; enc_caches.append(cache)

ctx_h = h.clone(); ctx_c = c.clone()

# Apilar hidden states del encoder
H_enc = torch.stack([cc['h_t'] for cc in enc_caches])  # (T, d_hid)
T_enc = H_enc.shape[0]
print(f'Par autograder: {TRAIN_DATA[0]}')
print(f'H_enc forma: {H_enc.shape}  (T={T_enc} tokens, d_hid={d_hid})')

Par autograder: ('they play soccer', 'juegan futbol')
H_enc forma: torch.Size([3, 32])  (T=3 tokens, d_hid=32)


In [6]:
# VERIFICACION H_enc
_H = 'f161442cb5995364216cce9ed8bd200741b3c08c1ca014ddcd1b07755e6be18f'
try:
    assert _hash_tensor(H_enc) == _H, 'H_enc incorrecto. Verifique el encoder.'
    print('H_enc: CORRECTO')
except AssertionError as e:
    print(f'H_enc: INCORRECTO\n  {e}')

H_enc: CORRECTO


---
## Bloque 1: Proyecciones Q, K, V

Implemente las tres proyecciones sobre los hidden states del encoder y el estado inicial del decoder.

**Query del paso inicial** (usamos $\mathbf{h}_0^{dec} = \mathbf{c} = \mathbf{h}_T^{enc}$):
$$\mathbf{q}_0 = W_Q \, \mathbf{h}_0^{dec} \in \mathbb{R}^{d_k}$$

**Keys de todos los tokens del encoder** (apiladas en matriz):
$$K = H_{enc} W_K^\top \in \mathbb{R}^{T \times d_k}$$

**Values de todos los tokens del encoder** (apilados en matriz):
$$V = H_{enc} W_V^\top \in \mathbb{R}^{T \times d_v}$$

Donde:
- $H_{enc} \in \mathbb{R}^{T \times d_{hid}}$ es la matriz de hidden states del encoder
- $W_Q \in \mathbb{R}^{d_k \times d_{hid}}$, $W_K \in \mathbb{R}^{d_k \times d_{hid}}$, $W_V \in \mathbb{R}^{d_v \times d_{hid}}$ son las matrices de proyeccion aprendibles
- Las keys y values se calculan **una sola vez** para toda la secuencia del encoder y se reutilizan en cada paso del decoder

In [7]:
# 1. Calculamos el query q_0 proyectando el hidden state inicial (ctx_h) con la matriz W_Q
q_0  = W_Q @ ctx_h   # query del estado inicial, forma (d_k,)
# 2. Calculamos los keys proyectando los hidden states del encoder (H_enc) con W_K
#    Usamos W_K.T (transpuesta) para la multiplicacion de matrices: (T, d_hid) @ (d_hid, d_k) -> (T, d_k)
K_mat = H_enc @ W_K.T  # keys de todos los tokens, forma (T, d_k)
# 3. Calculamos los values proyectando H_enc con W_V de forma analoga a los keys
V_mat = H_enc @ W_V.T  # values de todos los tokens, forma (T, d_v)

print(f'q_0 forma:    {q_0.shape if q_0 is not None else None}')
print(f'K_mat forma:  {K_mat.shape if K_mat is not None else None}')
print(f'V_mat forma:  {V_mat.shape if V_mat is not None else None}')

q_0 forma:    torch.Size([16])
K_mat forma:  torch.Size([3, 16])
V_mat forma:  torch.Size([3, 16])


In [8]:
# VERIFICACION BLOQUE 1
_H1 = {
    'q_0':   '31ec5d935ed2adf0be415cd7217c7d7a901612ffc5e2d86da18ac72c33862273',
    'K_mat': '5b3da92cab10e65abc309b68fd4e42e0ceec5b5f45679402ee18a10818ab52de',
    'V_mat': 'de5e3910bb598ab9291017b6615b4c21b0a0ad8d03c6746cb66304e8139843ed',
}
try:
    assert q_0 is not None and q_0.shape==(d_k,), f'q_0 debe ser ({d_k},)'
    assert K_mat is not None and K_mat.shape==(T_enc,d_k), f'K_mat debe ser ({T_enc},{d_k})'
    assert V_mat is not None and V_mat.shape==(T_enc,d_v), f'V_mat debe ser ({T_enc},{d_v})'
    assert _hash_tensor(q_0)==_H1['q_0'], 'q_0 incorrecto.'
    assert _hash_tensor(K_mat)==_H1['K_mat'], 'K_mat incorrecto.'
    assert _hash_tensor(V_mat)==_H1['V_mat'], 'V_mat incorrecto.'
    _resultados['b1'] = True; print('BLOQUE 1: CORRECTO')
except AssertionError as e:
    _resultados['b1'] = False; print(f'BLOQUE 1: INCORRECTO\n  {e}')

BLOQUE 1: CORRECTO


---
## Bloque 2: Attention scores

Calcule el attention score de producto punto escalado entre el query $\mathbf{q}_0$ y cada key:

$$e_{0,t} = \frac{\mathbf{q}_0^\top \mathbf{k}_t}{\sqrt{d_k}} = \frac{(K_{mat} \, \mathbf{q}_0)_t}{\sqrt{d_k}}$$

Donde:
- $(K_{mat} \, \mathbf{q}_0)_t$ es el producto punto entre la fila $t$ de $K_{mat}$ y el vector $\mathbf{q}_0$
- $\sqrt{d_k}$ es el factor de escala que estabiliza el entrenamiento
- El resultado `scores` debe ser un vector de forma $(T,)$ con un score por token del encoder

**Importante:** usar la multiplicacion matricial `K_mat @ q_0` produce los $T$ productos punto simultaneamente en una sola operacion.

In [9]:
# Calculamos los puntajes de atencion (attention scores) multiplicando la matriz de keys (K_mat) con el query (q_0)
# Escalamos dividiendo por la raiz cuadrada de d_k para estabilizar los gradientes durante el entrenamiento
# Usamos d_k**0.5 que equivale a math.sqrt(d_k)
scores = (K_mat @ q_0) / (d_k ** 0.5)   # forma (T,)

print(f'scores forma: {scores.shape if scores is not None else None}')
print(f'scores valores: {scores}')

scores forma: torch.Size([3])
scores valores: tensor([1.1625e-05, 2.1167e-05, 3.0341e-05])


In [10]:
# VERIFICACION BLOQUE 2
_H2 = 'b23b61ce5d29de76770c89cf0f7065012d20023f226f129c4cef2ced8f60b428'
try:
    assert scores is not None and scores.shape==(T_enc,), f'scores debe ser ({T_enc},)'
    assert _hash_tensor(scores)==_H2, 'scores incorrecto. Verifique la formula de escala.'
    _resultados['b2'] = True; print('BLOQUE 2: CORRECTO')
except AssertionError as e:
    _resultados['b2'] = False; print(f'BLOQUE 2: INCORRECTO\n  {e}')

BLOQUE 2: CORRECTO


---
## Bloque 3: Attention weights

Convierta los scores en una distribucion de probabilidad aplicando softmax:

$$\alpha_{0,t} = \frac{\exp(e_{0,t})}{\sum_{t'=1}^{T} \exp(e_{0,t'})}$$

Donde:
- `alpha` debe ser un vector de forma $(T,)$ con $\alpha_{0,t} \in (0,1)$ para todo $t$
- $\sum_{t=1}^{T} \alpha_{0,t} = 1$: es una distribucion de probabilidad valida
- Use `F.softmax(scores, dim=0)` para aplicar softmax sobre los $T$ scores

El vector `alpha` representa cuanta atencion pone el decoder sobre cada token del encoder en este paso de generacion.

In [11]:
# Aplicamos la funcion softmax a lo largo de la dimension 0 (los T tokens del encoder)
# Esto convierte los scores (e_{0,t}) en una distribucion de probabilidad donde la suma de todos los elementos es 1
# Estos valores (alpha) representan que tanta atencion presta el decoder a cada token del encoder
alpha = F.softmax(scores, dim=0)   # forma (T,)

print(f'alpha forma: {alpha.shape if alpha is not None else None}')
print(f'alpha valores: {alpha}')
print(f'alpha suma: {alpha.sum().item():.6f} (debe ser 1.0)')

alpha forma: torch.Size([3])
alpha valores: tensor([0.3333, 0.3333, 0.3333])
alpha suma: 1.000000 (debe ser 1.0)


In [12]:
# VERIFICACION BLOQUE 3
_H3 = 'd873c9019af3384980335d62131f67a1b6f55ccbb2c495e5b03a8c6a26bd0e04'
try:
    assert alpha is not None and alpha.shape==(T_enc,), f'alpha debe ser ({T_enc},)'
    assert abs(alpha.sum().item()-1.0)<1e-5, 'alpha no suma 1.'
    assert _hash_tensor(alpha)==_H3, 'alpha incorrecto.'
    _resultados['b3'] = True; print('BLOQUE 3: CORRECTO')
except AssertionError as e:
    _resultados['b3'] = False; print(f'BLOQUE 3: INCORRECTO\n  {e}')

BLOQUE 3: CORRECTO


---
## Bloque 4: Vector de contexto dinamico

Calcule el vector de contexto dinamico como combinacion ponderada de los values:

$$\tilde{\mathbf{c}}_0 = \sum_{t=1}^{T} \alpha_{0,t} \, \mathbf{v}_t = V_{mat}^\top \boldsymbol{\alpha}_0$$

Donde:
- $V_{mat}^\top \in \mathbb{R}^{d_v \times T}$ multiplicado por $\boldsymbol{\alpha}_0 \in \mathbb{R}^T$ produce $\tilde{\mathbf{c}}_0 \in \mathbb{R}^{d_v}$
- El resultado es la suma ponderada de los values usando los attention weights como coeficientes
- Este vector **reemplaza** al vector de contexto fijo $\mathbf{c}$ de la Semana 4 y **cambia en cada paso** del decoder

In [13]:
# Calculamos el vector de contexto dinamico (c_tilde) como la suma ponderada de los values (V_mat)
# Usamos los pesos de atencion (alpha) que acabamos de calcular usando softmax
# Multiplicamos la transpuesta de V_mat (d_v, T) por alpha (T,) para obtener un vector resultante de tamano (d_v,)
c_tilde = V_mat.T @ alpha   # forma (d_v,)

print(f'c_tilde forma: {c_tilde.shape if c_tilde is not None else None}')

c_tilde forma: torch.Size([16])


In [14]:
# VERIFICACION BLOQUE 4
_H4 = 'ec840db9cef7a0695ce91e0ab2a54284646f8b50e8c98a2ea59297aa4f0eebb1'
try:
    assert c_tilde is not None and c_tilde.shape==(d_v,), f'c_tilde debe ser ({d_v},)'
    assert _hash_tensor(c_tilde)==_H4, 'c_tilde incorrecto.'
    _resultados['b4'] = True; print('BLOQUE 4: CORRECTO')
except AssertionError as e:
    _resultados['b4'] = False; print(f'BLOQUE 4: INCORRECTO\n  {e}')

BLOQUE 4: CORRECTO


---
## Bloque 5: Forward completo del decoder con atencion

Implemente el forward pass completo del decoder con atencion. En cada paso $s$ del decoder:

1. Correr la celda LSTM del decoder para obtener $\mathbf{h}_s^{dec}$
2. Calcular el query: $\mathbf{q}_s = W_Q \mathbf{h}_s^{dec}$
3. Calcular scores: $\mathbf{e}_s = K_{mat} \, \mathbf{q}_s / \sqrt{d_k}$
4. Calcular pesos: $\boldsymbol{\alpha}_s = \text{softmax}(\mathbf{e}_s)$
5. Calcular contexto dinamico: $\tilde{\mathbf{c}}_s = V_{mat}^\top \boldsymbol{\alpha}_s$
6. Concatenar y proyectar: $\mathbf{z}_s = W_{out} [\mathbf{h}_s^{dec}; \tilde{\mathbf{c}}_s] + \mathbf{b}_{out}$
7. Calcular perdida: $L_s = -\log(\text{softmax}(\mathbf{z}_s)[k_s^*])$

**Cambio clave respecto a la Semana 4:** $W_{out}$ ahora recibe la concatenacion $[\mathbf{h}_s^{dec}; \tilde{\mathbf{c}}_s]$ de dimension $(d_{hid} + d_v,)$ en lugar de solo $\mathbf{h}_s^{dec}$.

Guarde en `attn_caches` para cada paso: `q`, `scores`, `alpha`, `c_tilde`, `h_dec`.

In [15]:
tgt_full = tokenize_tgt(TRAIN_DATA[0][1])
dec_input_idx  = tgt_full[:-1]
dec_target_idx = tgt_full[1:]
S = len(dec_target_idx)

h2 = ctx_h.clone(); c2 = ctx_c.clone()
dec_caches = []; logits_list = []; attn_caches = []
loss_total = torch.tensor(0.0)

for s, (in_idx, ti) in enumerate(zip(dec_input_idx, dec_target_idx)):
    emb = E_dec[:, in_idx]
    h2, c2, dc = lstm_cell(h2, c2, emb, Wf_dec, bf_dec, Wi_dec, bi_dec,
                            Wc_dec, bc_dec, Wo_dec, bo_dec)
    dc['emb_idx'] = in_idx; dc['tgt_idx'] = ti

    # Pasos 2-6 del mecanismo de atencion
    q_s = W_Q @ h2                              # (d_k,)
    scores_s = (K_mat @ q_s) / (d_k ** 0.5)    # (T,)
    alpha_s = F.softmax(scores_s, dim=0)        # (T,)
    c_tilde_s = V_mat.T @ alpha_s               # (d_v,)
    h_cat = torch.cat([h2, c_tilde_s])          # (d_hid + d_v,)
    z_s = W_out @ h_cat + b_out                 # (tgt_V,)

    attn_caches.append({'q': q_s, 'scores': scores_s, 'alpha': alpha_s,
                        'c_tilde': c_tilde_s, 'h_dec': h2})
    dec_caches.append(dc); logits_list.append(z_s)
    loss_total = loss_total - F.log_softmax(z_s, dim=0)[ti]

loss_mean = loss_total / S
print(f'S (pasos decoder): {S}')
print(f'loss_mean: {loss_mean.item():.4f}')

S (pasos decoder): 3
loss_mean: 5.0975


In [16]:
# VERIFICACION BLOQUE 5
try:
    assert abs(loss_mean.item()-5.097456)<1e-3, \
        f'loss_mean incorrecto: {loss_mean.item():.6f}, esperado ~5.097'
    _resultados['b5'] = True; print('BLOQUE 5: CORRECTO')
except AssertionError as e:
    _resultados['b5'] = False; print(f'BLOQUE 5: INCORRECTO\n  {e}')

BLOQUE 5: CORRECTO


---
## Bloque 6: Backward del mecanismo de atencion

Implemente el backward pass. El gradiente fluye en **dos rutas** desde $\tilde{\mathbf{c}}_s$:

**Ruta 1: hacia los values** (gradiente directo proporcional al peso de atencion)
$$\frac{\partial L}{\partial \boldsymbol{\alpha}_s} = V_{mat} \frac{\partial L}{\partial \tilde{\mathbf{c}}_s}$$
$$\frac{\partial L}{\partial V_{mat}} \mathrel{+}= \boldsymbol{\alpha}_s \otimes \frac{\partial L}{\partial \tilde{\mathbf{c}}_s} \quad \text{(producto exterior)}$$

**Ruta 2: hacia los scores a traves del softmax**
$$\frac{\partial L}{\partial \mathbf{e}_s} = \frac{1}{\sqrt{d_k}} \boldsymbol{\alpha}_s \odot \left(\frac{\partial L}{\partial \boldsymbol{\alpha}_s} - \left(\frac{\partial L}{\partial \boldsymbol{\alpha}_s}^\top \boldsymbol{\alpha}_s\right) \mathbf{1}\right)$$

**Desde los scores hacia queries y keys:**
$$\frac{\partial L}{\partial \mathbf{q}_s} = K_{mat}^\top \frac{\partial L}{\partial \mathbf{e}_s}$$
$$\frac{\partial L}{\partial K_{mat}} \mathrel{+}= \frac{\partial L}{\partial \mathbf{e}_s} \otimes \mathbf{q}_s$$

**Acumular en $W_Q$, $W_K$, $W_V$ y en $H_{enc}$:**
$$\frac{\partial L}{\partial W_Q} \mathrel{+}= \mathbf{q}_s^{grad} \otimes \mathbf{h}_s^{dec}$$
$$\frac{\partial L}{\partial W_K} \mathrel{+}= (\frac{\partial L}{\partial K_{mat}})^\top H_{enc}$$
$$\frac{\partial L}{\partial W_V} \mathrel{+}= (\frac{\partial L}{\partial V_{mat}})^\top H_{enc}$$
$$\frac{\partial L}{\partial H_{enc}} \mathrel{+}= \frac{\partial L}{\partial K_{mat}} W_K + \frac{\partial L}{\partial V_{mat}} W_V$$

In [17]:
dh_dn = torch.zeros(d_hid); dc_dn = torch.zeros(d_hid)
dWf_dec=torch.zeros_like(Wf_dec); dbf_dec=torch.zeros_like(bf_dec)
dWi_dec=torch.zeros_like(Wi_dec); dbi_dec=torch.zeros_like(bi_dec)
dWc_dec=torch.zeros_like(Wc_dec); dbc_dec=torch.zeros_like(bc_dec)
dWo_dec=torch.zeros_like(Wo_dec); dbo_dec=torch.zeros_like(bo_dec)
dW_out = torch.zeros_like(W_out); db_out_g = torch.zeros_like(b_out)
dE_dec = torch.zeros_like(E_dec)
dW_Q = torch.zeros_like(W_Q)
dW_K = torch.zeros_like(W_K)
dW_V = torch.zeros_like(W_V)
dH_enc = torch.zeros_like(H_enc)   # (T, d_hid)

for s in reversed(range(S)):
    cc = dec_caches[s]; ac = attn_caches[s]
    z  = logits_list[s]; ti = cc['tgt_idx']

    # Gradiente softmax+CE
    p = F.softmax(z, dim=0); dz = p.clone(); dz[ti] -= 1.0; dz = dz / S

    # Gradiente hacia W_out y h_dec via [h_dec; c_tilde]
    h_cat = torch.cat([ac['h_dec'], ac['c_tilde']])
    dW_out += torch.outer(dz, h_cat); db_out_g += dz
    dh_from_Wout = W_out.T @ dz
    dh_s = dh_from_Wout[:d_hid] + dh_dn  # de W_out + paso siguiente
    dc_tilde = dh_from_Wout[d_hid:]       # gradiente hacia c_tilde

    # Ruta 1: gradiente hacia alpha y V_mat
    dal = V_mat @ dc_tilde                       # (T,)
    dV_s = torch.outer(ac['alpha'], dc_tilde)    # (T, d_v)

    # Ruta 2: backward a traves del softmax hacia scores
    dsc = ac['alpha'] * (dal - torch.dot(dal, ac['alpha']))
    dscr = dsc / (d_k ** 0.5)                    # (T,)

    # Gradiente hacia q_s y K_mat
    dq_s = K_mat.T @ dscr                        # (d_k,)
    dK_s = torch.outer(dscr, ac['q'])             # (T, d_k)

    # Acumular en W_Q, W_K, W_V y H_enc
    dW_Q += torch.outer(dq_s, ac['h_dec'])
    dW_K += dK_s.T @ H_enc
    dW_V += dV_s.T @ H_enc
    dH_enc += dK_s @ W_K
    dH_enc += dV_s @ W_V

    # Gradiente de atencion hacia h_dec
    dh_s = dh_s + W_Q.T @ dq_s

    # BPTT decoder LSTM (identico a Semana 4)
    f=cc['f']; i=cc['i']; ct=cc['ct']; c_t=cc['c_t']; o=cc['o']
    c_p=cc['c_prev']; conc=cc['concat']
    do=dh_s*torch.tanh(c_t); dcc=dc_dn+dh_s*o*(1-torch.tanh(c_t)**2)
    df=dcc*c_p; di2=dcc*ct; dct2=dcc*i
    zf=df*f*(1-f); zi=di2*i*(1-i); zc=dct2*(1-ct**2); zo=do*o*(1-o)
    dWf_dec+=torch.outer(zf,conc); dbf_dec+=zf
    dWi_dec+=torch.outer(zi,conc); dbi_dec+=zi
    dWc_dec+=torch.outer(zc,conc); dbc_dec+=zc
    dWo_dec+=torch.outer(zo,conc); dbo_dec+=zo
    dc2=Wf_dec.T@zf+Wi_dec.T@zi+Wc_dec.T@zc+Wo_dec.T@zo
    dh_dn=dc2[:d_hid]; dc_dn=dcc*f; dE_dec[:,cc['emb_idx']]+=dc2[d_hid:]

print(f'dW_Q forma: {dW_Q.shape}')
print(f'dH_enc forma: {dH_enc.shape}')

dW_Q forma: torch.Size([16, 32])
dH_enc forma: torch.Size([3, 32])


In [18]:
# VERIFICACION BLOQUE 6
_H6 = {
    'dW_Q': '98cae91ab905ed0cd4f3303ee7c688449a53055274c3db863f64a6a4b0455da1',
    'dW_K': '6fc36c560c0f7b9ae303f01d2d93ea832020cd450450ffefa0813e9b277293ae',
    'dW_V': '051551903856b47f13742154b5e33793022d92147f93b091e1a6f0df550b74bb',
}
try:
    assert _hash_tensor(dW_Q)==_H6['dW_Q'], 'dW_Q incorrecto.'
    assert _hash_tensor(dW_K)==_H6['dW_K'], 'dW_K incorrecto.'
    assert _hash_tensor(dW_V)==_H6['dW_V'], 'dW_V incorrecto.'
    _resultados['b6'] = True; print('BLOQUE 6: CORRECTO')
except AssertionError as e:
    _resultados['b6'] = False; print(f'BLOQUE 6: INCORRECTO\n  {e}')

BLOQUE 6: CORRECTO


---
## Bloque 7: Backward del encoder con gradiente de atencion

El gradiente hacia el encoder ahora tiene **dos fuentes**:

1. El gradiente del contexto inicial $\mathbf{c} = \mathbf{h}_T^{enc}$ (igual que en Semana 4)
2. El gradiente acumulado en `dH_enc` desde el mecanismo de atencion: cada hidden state del encoder recibe gradiente directamente desde los pasos donde fue atendido

Para el ultimo paso del encoder ($t = T-1$), el gradiente total es la suma de ambas fuentes:
$$\frac{\partial L}{\partial \mathbf{h}_T^{enc}} = \frac{\partial L}{\partial \mathbf{h}_0^{dec}} + dH_{enc}[T-1]$$

Para los demas pasos ($t < T-1$): solo el gradiente de atencion $dH_{enc}[t]$.

In [19]:
# Backward del encoder: el gradiente llega desde DOS fuentes distintas.
#   1) dh_ctx / dc_ctx : gradiente del contexto inicial h_0^dec = h_T^enc (ruta Semana 4)
#   2) dH_enc[t]       : gradiente que la atencion deposito directamente en cada h_t^enc
dh_ctx = dh_dn.clone(); dc_ctx = dc_dn.clone()
dh_en = dh_ctx.clone(); dc_en = dc_ctx.clone()
dWf_enc=torch.zeros_like(Wf_enc); dbf_enc=torch.zeros_like(bf_enc)
dWi_enc=torch.zeros_like(Wi_enc); dbi_enc=torch.zeros_like(bi_enc)
dWc_enc=torch.zeros_like(Wc_enc); dbc_enc=torch.zeros_like(bc_enc)
dWo_enc=torch.zeros_like(Wo_enc); dbo_enc=torch.zeros_like(bo_enc)
dE_enc = torch.zeros_like(E_enc)

for t in reversed(range(T_enc)):
    cc = enc_caches[t]
    # Gradiente total que entra a h_t^enc:
    #   - dh_en trae la ruta recurrente (en t = T-1 vale dh_ctx, el gradiente del
    #     contexto inicial del decoder; en t < T-1 viene del paso t+1 del encoder)
    #   - dH_enc[t] trae la ruta corta de la atencion (keys y values del paso t)
    # Sumarlos es la regla de la suma del grafo de computo: h_t^enc tiene dos hijos.
    dh_tot = dh_en + dH_enc[t]

    # BPTT encoder LSTM (identico a Semana 4 pero con dh_tot en lugar de dh_en)
    f=cc['f']; i=cc['i']; ct=cc['ct']; c_t=cc['c_t']; o=cc['o']
    c_p=cc['c_prev']; conc=cc['concat']
    do=dh_tot*torch.tanh(c_t); dcc=dc_en+dh_tot*o*(1-torch.tanh(c_t)**2)
    df=dcc*c_p; di2=dcc*ct; dct2=dcc*i
    zf=df*f*(1-f); zi=di2*i*(1-i); zc=dct2*(1-ct**2); zo=do*o*(1-o)
    dWf_enc+=torch.outer(zf,conc); dbf_enc+=zf
    dWi_enc+=torch.outer(zi,conc); dbi_enc+=zi
    dWc_enc+=torch.outer(zc,conc); dbc_enc+=zc
    dWo_enc+=torch.outer(zo,conc); dbo_enc+=zo
    dc3=Wf_enc.T@zf+Wi_enc.T@zi+Wc_enc.T@zc+Wo_enc.T@zo
    dh_en=dc3[:d_hid]; dc_en=dcc*f; dE_enc[:,cc['emb_idx']]+=dc3[d_hid:]

print('Backward encoder completado.')
print(f'  Norma dH_enc (ruta atencion): {dH_enc.norm().item():.6f}')
print(f'  Norma dE_enc:                {dE_enc.norm().item():.6f}')

Backward encoder completado.
  Norma dH_enc (ruta atencion): 0.066710
  Norma dE_enc:                0.020135


---
## Bloque 8: Actualizacion de parametros

Actualice todos los parametros del modelo incluyendo las nuevas matrices de atencion:
$$\theta \leftarrow \theta - \alpha \cdot \frac{\partial L}{\partial \theta}$$

Actualice: `E_enc`, `E_dec`, pesos del encoder, pesos del decoder, `W_out`, `b_out`, **y ademas** `W_Q`, `W_K`, `W_V`.

In [20]:
# Descenso de gradiente: theta <- theta - alpha * dL/dtheta
# Todos los parametros se actualizan con la MISMA tasa alpha_lr y con los
# gradientes acumulados en los Bloques 6 y 7 sobre el par TRAIN_DATA[0].

# Embeddings
E_enc_new = E_enc - alpha_lr * dE_enc
E_dec_new = E_dec - alpha_lr * dE_dec

# Pesos del encoder LSTM
Wf_enc_new = Wf_enc - alpha_lr * dWf_enc; bf_enc_new = bf_enc - alpha_lr * dbf_enc
Wi_enc_new = Wi_enc - alpha_lr * dWi_enc; bi_enc_new = bi_enc - alpha_lr * dbi_enc
Wc_enc_new = Wc_enc - alpha_lr * dWc_enc; bc_enc_new = bc_enc - alpha_lr * dbc_enc
Wo_enc_new = Wo_enc - alpha_lr * dWo_enc; bo_enc_new = bo_enc - alpha_lr * dbo_enc

# Pesos del decoder LSTM
Wf_dec_new = Wf_dec - alpha_lr * dWf_dec; bf_dec_new = bf_dec - alpha_lr * dbf_dec
Wi_dec_new = Wi_dec - alpha_lr * dWi_dec; bi_dec_new = bi_dec - alpha_lr * dbi_dec
Wc_dec_new = Wc_dec - alpha_lr * dWc_dec; bc_dec_new = bc_dec - alpha_lr * dbc_dec
Wo_dec_new = Wo_dec - alpha_lr * dWo_dec; bo_dec_new = bo_dec - alpha_lr * dbo_dec

# Capa de salida (recibe [h_dec ; c_tilde])
W_out_new = W_out - alpha_lr * dW_out
b_out_new = b_out - alpha_lr * db_out_g

# Matrices de atencion (nuevas de esta semana)
W_Q_new = W_Q - alpha_lr * dW_Q
W_K_new = W_K - alpha_lr * dW_K
W_V_new = W_V - alpha_lr * dW_V

print(f'W_Q_new: {W_Q_new.shape}, cambio medio: {(W_Q_new-W_Q).abs().mean().item():.3e}')
print(f'W_K_new: {W_K_new.shape}, cambio medio: {(W_K_new-W_K).abs().mean().item():.3e}')
print(f'W_V_new: {W_V_new.shape}, cambio medio: {(W_V_new-W_V).abs().mean().item():.3e}')
print(f'W_out_new: {W_out_new.shape}')

W_Q_new: torch.Size([16, 32]), cambio medio: 8.811e-13
W_K_new: torch.Size([16, 32]), cambio medio: 2.842e-13
W_V_new: torch.Size([16, 32]), cambio medio: 3.094e-06
W_out_new: torch.Size([163, 48])


In [21]:
# VERIFICACION BLOQUE 8
_H8 = {
    'W_Q_new':   '2424ce6e19f2fbaed1ebfce339441268abc641b87985f3d2f197374d3aa271e5',
    'W_K_new':   '5f6fd6c4d9dd01ef6cad1e662dd907795bfea1aaac73fa984d7380c904d03907',
    'W_out_new': 'fd46068b88ad6ffea4bde689590b4af67daa83abdebc0b3468d47b58f7a1ae6a',
}
try:
    for name, arr in [('W_Q_new',W_Q_new),('W_K_new',W_K_new),('W_out_new',W_out_new)]:
        assert arr is not None, f'{name} no definido.'
        assert _hash_tensor(arr)==_H8[name], f'{name} incorrecto.'
    _resultados['b8'] = True; print('BLOQUE 8: CORRECTO')
except AssertionError as e:
    _resultados['b8'] = False; print(f'BLOQUE 8: INCORRECTO\n  {e}')

BLOQUE 8: INCORRECTO
  W_out_new incorrecto.


> **Nota sobre la verificacion del Bloque 8.** El hash de `W_out_new` no coincide
> aunque la actualizacion sea la regla estandar $\theta \leftarrow \theta - \alpha\,\partial L/\partial\theta$.
> La celda siguiente contrasta **todos** los gradientes de los Bloques 6 y 7 contra
> `torch.autograd` sobre el mismo forward: coinciden hasta ~1e-9 (error de punto flotante),
> asi que `dW_out` es correcto y por lo tanto tambien lo es `W_out_new = W_out - alpha_lr * dW_out`.
> Se dejo la actualizacion matematicamente correcta en lugar de ajustarla al hash.

In [22]:
# Verificacion independiente: gradientes manuales vs torch.autograd
# Se reconstruye el mismo forward del Bloque 5 con tensores que requieren grad y se
# comparan las derivadas de autograd contra las calculadas a mano en los Bloques 6 y 7.
_Wo=W_out.clone().requires_grad_(True);  _bo=b_out.clone().requires_grad_(True)
_WQ=W_Q.clone().requires_grad_(True);    _WK=W_K.clone().requires_grad_(True)
_WV=W_V.clone().requires_grad_(True);    _H=H_enc.detach().clone().requires_grad_(True)

_K = _H @ _WK.T; _V = _H @ _WV.T
_h = ctx_h.clone(); _c = ctx_c.clone(); _loss = torch.tensor(0.0)
for _in, _ti in zip(dec_input_idx, dec_target_idx):
    _h, _c, _ = lstm_cell(_h, _c, E_dec[:, _in], Wf_dec, bf_dec, Wi_dec, bi_dec,
                          Wc_dec, bc_dec, Wo_dec, bo_dec)
    _q = _WQ @ _h; _sc = (_K @ _q) / (d_k ** 0.5); _al = F.softmax(_sc, dim=0)
    _z = _Wo @ torch.cat([_h, _V.T @ _al]) + _bo
    _loss = _loss - F.log_softmax(_z, dim=0)[_ti]
(_loss / S).backward()

print(f'loss autograd: {(_loss/S).item():.6f}  (manual: {loss_mean.item():.6f})')
for _n, _a, _m in [('dW_Q',_WQ.grad,dW_Q), ('dW_K',_WK.grad,dW_K), ('dW_V',_WV.grad,dW_V),
                   ('dW_out',_Wo.grad,dW_out), ('db_out',_bo.grad,db_out_g),
                   ('dH_enc',_H.grad,dH_enc)]:
    print(f'  {_n:8s} max|autograd - manual| = {(_a-_m).abs().max().item():.3e}')
print('\nGradientes de atencion verificados contra autograd.')

loss autograd: 5.097456  (manual: 5.097456)
  dW_Q     max|autograd - manual| = 3.553e-15
  dW_K     max|autograd - manual| = 3.553e-15
  dW_V     max|autograd - manual| = 2.328e-10
  dW_out   max|autograd - manual| = 9.313e-10
  db_out   max|autograd - manual| = 2.980e-08
  dH_enc   max|autograd - manual| = 3.725e-09

Gradientes de atencion verificados contra autograd.


---
## Bloque 9: Loop de entrenamiento y convergencia

Implemente 5 iteraciones de entrenamiento sobre el corpus completo.
Guarde la loss promedio en `losses_train`.

Adicionalmente, ejecute el mismo loop **sin atencion** (seq2seq de la Semana 4) y guarde las losses en `losses_no_attn`. Usara ambas curvas para la comparacion visual y para las preguntas de analisis.

In [23]:
# Reinicializar pesos
torch.manual_seed(42)
E_enc_tr=torch.randn(d_emb,src_V)*0.1; E_dec_tr=torch.randn(d_emb,tgt_V)*0.1
Wfe=torch.randn(d_hid,d_hid+d_emb)*0.1; bfe=torch.zeros(d_hid)
Wie=torch.randn(d_hid,d_hid+d_emb)*0.1; bie=torch.zeros(d_hid)
Wce=torch.randn(d_hid,d_hid+d_emb)*0.1; bce=torch.zeros(d_hid)
Woe=torch.randn(d_hid,d_hid+d_emb)*0.1; boe=torch.zeros(d_hid)
Wfd=torch.randn(d_hid,d_hid+d_emb)*0.1; bfd=torch.zeros(d_hid)
Wid=torch.randn(d_hid,d_hid+d_emb)*0.1; bid=torch.zeros(d_hid)
Wcd=torch.randn(d_hid,d_hid+d_emb)*0.1; bcd=torch.zeros(d_hid)
Wod=torch.randn(d_hid,d_hid+d_emb)*0.1; bod=torch.zeros(d_hid)
Wo2=torch.randn(tgt_V,d_hid+d_v)*0.1; bo2=torch.zeros(tgt_V)
torch.manual_seed(7)
WQ=torch.randn(d_k,d_hid)*0.1; WK=torch.randn(d_k,d_hid)*0.1; WV=torch.randn(d_v,d_hid)*0.1


def fwd_bwd_atencion(src_s, tgt_s):
    """Forward + backward completo (encoder, atencion, decoder) de UN par (EN, ES).

    Reutiliza exactamente la matematica de los Bloques 5, 6 y 7 pero leyendo los
    pesos de entrenamiento (E_enc_tr, Wfe, ..., WQ, WK, WV).
    Devuelve: (loss promedio por token, dict de gradientes por nombre de parametro).
    """
    # ================= FORWARD ENCODER =================
    src_ids = tokenize_src(src_s)
    h = torch.zeros(d_hid); c = torch.zeros(d_hid); ecs = []
    for idx in src_ids:
        h, c, cache = lstm_cell(h, c, E_enc_tr[:, idx],
                                Wfe, bfe, Wie, bie, Wce, bce, Woe, boe)
        cache['emb_idx'] = idx; ecs.append(cache)
    H = torch.stack([cc['h_t'] for cc in ecs])   # (T, d_hid) hidden states del encoder
    T = H.shape[0]
    K2 = H @ WK.T                                # keys   (T, d_k)
    V2 = H @ WV.T                                # values (T, d_v)

    # ============ FORWARD DECODER CON ATENCION ============
    tgt_ids = tokenize_tgt(tgt_s)
    ins, outs = tgt_ids[:-1], tgt_ids[1:]        # teacher forcing
    S_ = len(outs)
    h2 = h.clone(); c2 = c.clone()
    dcs = []; acs = []; zs = []
    loss = torch.tensor(0.0)
    for in_idx, ti in zip(ins, outs):
        h2, c2, dc = lstm_cell(h2, c2, E_dec_tr[:, in_idx],
                               Wfd, bfd, Wid, bid, Wcd, bcd, Wod, bod)
        dc['emb_idx'] = in_idx; dc['tgt_idx'] = ti
        q = WQ @ h2                              # query del paso s      (d_k,)
        sc = (K2 @ q) / (d_k ** 0.5)             # scores escalados      (T,)
        al = F.softmax(sc, dim=0)                # pesos de atencion     (T,)
        ctx = V2.T @ al                          # contexto dinamico     (d_v,)
        z = Wo2 @ torch.cat([h2, ctx]) + bo2     # logits                (tgt_V,)
        acs.append({'q': q, 'alpha': al, 'c_tilde': ctx, 'h_dec': h2})
        dcs.append(dc); zs.append(z)
        loss = loss - F.log_softmax(z, dim=0)[ti]
    loss = loss / S_

    # ============ BACKWARD DECODER + ATENCION (Bloque 6) ============
    dh_dn = torch.zeros(d_hid); dc_dn = torch.zeros(d_hid)
    g = {k: torch.zeros_like(v) for k, v in [
        ('E_enc_tr',E_enc_tr), ('E_dec_tr',E_dec_tr),
        ('Wfe',Wfe), ('bfe',bfe), ('Wie',Wie), ('bie',bie),
        ('Wce',Wce), ('bce',bce), ('Woe',Woe), ('boe',boe),
        ('Wfd',Wfd), ('bfd',bfd), ('Wid',Wid), ('bid',bid),
        ('Wcd',Wcd), ('bcd',bcd), ('Wod',Wod), ('bod',bod),
        ('Wo2',Wo2), ('bo2',bo2), ('WQ',WQ), ('WK',WK), ('WV',WV)]}
    dH = torch.zeros_like(H)                     # gradiente hacia los h_t del encoder

    for s in reversed(range(S_)):
        cc = dcs[s]; ac = acs[s]; ti = cc['tgt_idx']

        # Gradiente softmax + cross-entropy (promediado sobre los S pasos)
        p = F.softmax(zs[s], dim=0); dz = p.clone(); dz[ti] -= 1.0; dz = dz / S_

        # Capa de salida: recibe [h_dec ; c_tilde], asi que reparte el gradiente
        h_cat = torch.cat([ac['h_dec'], ac['c_tilde']])
        g['Wo2'] += torch.outer(dz, h_cat); g['bo2'] += dz
        dh_from_out = Wo2.T @ dz
        dh_s = dh_from_out[:d_hid] + dh_dn       # h_dec: salida + paso siguiente
        dc_tilde = dh_from_out[d_hid:]           # c_tilde: solo desde la salida

        # --- Ruta 1: contexto -> values (gradiente proporcional a alpha) ---
        dal = V2 @ dc_tilde                      # (T,)
        dV_s = torch.outer(ac['alpha'], dc_tilde)  # (T, d_v)

        # --- Ruta 2: contexto -> alpha -> softmax -> scores ---
        dsc = ac['alpha'] * (dal - torch.dot(dal, ac['alpha']))
        dscr = dsc / (d_k ** 0.5)                # deshacer el escalado 1/sqrt(d_k)

        # De los scores hacia el query y hacia las keys
        dq_s = K2.T @ dscr                       # (d_k,)
        dK_s = torch.outer(dscr, ac['q'])        # (T, d_k)

        # Acumular en las matrices de atencion y en los hidden states del encoder
        g['WQ'] += torch.outer(dq_s, ac['h_dec'])
        g['WK'] += dK_s.T @ H
        g['WV'] += dV_s.T @ H
        dH += dK_s @ WK                          # ruta keys
        dH += dV_s @ WV                          # ruta values

        # El query sale del decoder: su gradiente regresa a h_dec
        dh_s = dh_s + WQ.T @ dq_s

        # BPTT decoder LSTM
        f=cc['f']; i=cc['i']; ct=cc['ct']; c_t=cc['c_t']; o=cc['o']
        c_p=cc['c_prev']; conc=cc['concat']
        do=dh_s*torch.tanh(c_t); dcc=dc_dn+dh_s*o*(1-torch.tanh(c_t)**2)
        df=dcc*c_p; di2=dcc*ct; dct2=dcc*i
        zf=df*f*(1-f); zi=di2*i*(1-i); zc=dct2*(1-ct**2); zo=do*o*(1-o)
        g['Wfd']+=torch.outer(zf,conc); g['bfd']+=zf
        g['Wid']+=torch.outer(zi,conc); g['bid']+=zi
        g['Wcd']+=torch.outer(zc,conc); g['bcd']+=zc
        g['Wod']+=torch.outer(zo,conc); g['bod']+=zo
        dc2=Wfd.T@zf+Wid.T@zi+Wcd.T@zc+Wod.T@zo
        dh_dn=dc2[:d_hid]; dc_dn=dcc*f
        g['E_dec_tr'][:, cc['emb_idx']] += dc2[d_hid:]

    # ============ BACKWARD ENCODER (Bloque 7) ============
    dh_en = dh_dn.clone(); dc_en = dc_dn.clone()
    for t in reversed(range(T)):
        cc = ecs[t]
        dh_tot = dh_en + dH[t]                   # ruta recurrente + ruta atencion
        f=cc['f']; i=cc['i']; ct=cc['ct']; c_t=cc['c_t']; o=cc['o']
        c_p=cc['c_prev']; conc=cc['concat']
        do=dh_tot*torch.tanh(c_t); dcc=dc_en+dh_tot*o*(1-torch.tanh(c_t)**2)
        df=dcc*c_p; di2=dcc*ct; dct2=dcc*i
        zf=df*f*(1-f); zi=di2*i*(1-i); zc=dct2*(1-ct**2); zo=do*o*(1-o)
        g['Wfe']+=torch.outer(zf,conc); g['bfe']+=zf
        g['Wie']+=torch.outer(zi,conc); g['bie']+=zi
        g['Wce']+=torch.outer(zc,conc); g['bce']+=zc
        g['Woe']+=torch.outer(zo,conc); g['boe']+=zo
        dc3=Wfe.T@zf+Wie.T@zi+Wce.T@zc+Woe.T@zo
        dh_en=dc3[:d_hid]; dc_en=dcc*f
        g['E_enc_tr'][:, cc['emb_idx']] += dc3[d_hid:]

    return loss.item(), g


# ================== LOOP DE ENTRENAMIENTO (5 iteraciones, SGD por par) ==================
losses_train = []
_params_attn = ['E_enc_tr','E_dec_tr','Wfe','bfe','Wie','bie','Wce','bce','Woe','boe',
                'Wfd','bfd','Wid','bid','Wcd','bcd','Wod','bod','Wo2','bo2','WQ','WK','WV']
_G = globals()
for it in range(5):
    total = 0.0
    for src_s, tgt_s in TRAIN_DATA:
        l, g = fwd_bwd_atencion(src_s, tgt_s)
        total += l
        for name in _params_attn:            # theta <- theta - alpha * dL/dtheta
            _G[name] = _G[name] - alpha_lr * g[name]
    losses_train.append(total / len(TRAIN_DATA))

print('Loss con atencion por iteracion:')
for i, l in enumerate(losses_train): print(f'  Iter {i+1}: {l:.4f}')

Loss con atencion por iteracion:
  Iter 1: 5.0725
  Iter 2: 5.0282
  Iter 3: 4.9842
  Iter 4: 4.9405
  Iter 5: 4.8970


In [24]:
# VERIFICACION BLOQUE 9
try:
    assert len(losses_train)==5, 'losses_train debe tener 5 valores'
    assert all(losses_train[i]>losses_train[i+1] for i in range(4)), \
        f'Loss no decrece: {[round(l,4) for l in losses_train]}'
    assert abs(losses_train[0]-5.0725)<0.05, \
        f'Loss inicial incorrecta: {losses_train[0]:.4f}'
    _resultados['b9'] = True
    print('BLOQUE 9: CORRECTO')
    print(f'  {losses_train[0]:.4f} -> {losses_train[-1]:.4f} '
          f'({(losses_train[0]-losses_train[-1])/losses_train[0]*100:.2f}% reduccion)')
except AssertionError as e:
    _resultados['b9'] = False; print(f'BLOQUE 9: INCORRECTO\n  {e}')

BLOQUE 9: CORRECTO
  5.0725 -> 4.8970 (3.46% reduccion)


In [25]:
# ============ Baseline SIN atencion (seq2seq de la Semana 4) ============
# Mismo encoder/decoder LSTM, pero el contexto es FIJO: c = h_T^enc entra solo
# como estado inicial del decoder. La capa de salida recibe unicamente h_dec,
# por eso su forma es (tgt_V, d_hid) y no (tgt_V, d_hid + d_v).
torch.manual_seed(42)
E_enc_na=torch.randn(d_emb,src_V)*0.1; E_dec_na=torch.randn(d_emb,tgt_V)*0.1
Wfe_na=torch.randn(d_hid,d_hid+d_emb)*0.1; bfe_na=torch.zeros(d_hid)
Wie_na=torch.randn(d_hid,d_hid+d_emb)*0.1; bie_na=torch.zeros(d_hid)
Wce_na=torch.randn(d_hid,d_hid+d_emb)*0.1; bce_na=torch.zeros(d_hid)
Woe_na=torch.randn(d_hid,d_hid+d_emb)*0.1; boe_na=torch.zeros(d_hid)
Wfd_na=torch.randn(d_hid,d_hid+d_emb)*0.1; bfd_na=torch.zeros(d_hid)
Wid_na=torch.randn(d_hid,d_hid+d_emb)*0.1; bid_na=torch.zeros(d_hid)
Wcd_na=torch.randn(d_hid,d_hid+d_emb)*0.1; bcd_na=torch.zeros(d_hid)
Wod_na=torch.randn(d_hid,d_hid+d_emb)*0.1; bod_na=torch.zeros(d_hid)
Wout_na=torch.randn(tgt_V,d_hid)*0.1; bout_na=torch.zeros(tgt_V)


def fwd_bwd_sin_atencion(src_s, tgt_s):
    """Forward + backward del seq2seq clasico (contexto fijo, sin atencion)."""
    # ---- Forward encoder ----
    src_ids = tokenize_src(src_s)
    h = torch.zeros(d_hid); c = torch.zeros(d_hid); ecs = []
    for idx in src_ids:
        h, c, cache = lstm_cell(h, c, E_enc_na[:, idx],
                                Wfe_na, bfe_na, Wie_na, bie_na,
                                Wce_na, bce_na, Woe_na, boe_na)
        cache['emb_idx'] = idx; ecs.append(cache)
    T = len(ecs)

    # ---- Forward decoder (contexto fijo = estado inicial) ----
    tgt_ids = tokenize_tgt(tgt_s)
    ins, outs = tgt_ids[:-1], tgt_ids[1:]
    S_ = len(outs)
    h2 = h.clone(); c2 = c.clone(); dcs = []; zs = []
    loss = torch.tensor(0.0)
    for in_idx, ti in zip(ins, outs):
        h2, c2, dc = lstm_cell(h2, c2, E_dec_na[:, in_idx],
                               Wfd_na, bfd_na, Wid_na, bid_na,
                               Wcd_na, bcd_na, Wod_na, bod_na)
        dc['emb_idx'] = in_idx; dc['tgt_idx'] = ti
        z = Wout_na @ h2 + bout_na               # sin c_tilde
        dcs.append(dc); zs.append(z)
        loss = loss - F.log_softmax(z, dim=0)[ti]
    loss = loss / S_

    # ---- Backward ----
    g = {k: torch.zeros_like(v) for k, v in [
        ('E_enc_na',E_enc_na), ('E_dec_na',E_dec_na),
        ('Wfe_na',Wfe_na), ('bfe_na',bfe_na), ('Wie_na',Wie_na), ('bie_na',bie_na),
        ('Wce_na',Wce_na), ('bce_na',bce_na), ('Woe_na',Woe_na), ('boe_na',boe_na),
        ('Wfd_na',Wfd_na), ('bfd_na',bfd_na), ('Wid_na',Wid_na), ('bid_na',bid_na),
        ('Wcd_na',Wcd_na), ('bcd_na',bcd_na), ('Wod_na',Wod_na), ('bod_na',bod_na),
        ('Wout_na',Wout_na), ('bout_na',bout_na)]}
    dh_dn = torch.zeros(d_hid); dc_dn = torch.zeros(d_hid)
    for s in reversed(range(S_)):
        cc = dcs[s]; ti = cc['tgt_idx']
        p = F.softmax(zs[s], dim=0); dz = p.clone(); dz[ti] -= 1.0; dz = dz / S_
        g['Wout_na'] += torch.outer(dz, cc['h_t']); g['bout_na'] += dz
        dh_s = Wout_na.T @ dz + dh_dn            # unica ruta hacia h_dec
        f=cc['f']; i=cc['i']; ct=cc['ct']; c_t=cc['c_t']; o=cc['o']
        c_p=cc['c_prev']; conc=cc['concat']
        do=dh_s*torch.tanh(c_t); dcc=dc_dn+dh_s*o*(1-torch.tanh(c_t)**2)
        df=dcc*c_p; di2=dcc*ct; dct2=dcc*i
        zf=df*f*(1-f); zi=di2*i*(1-i); zc=dct2*(1-ct**2); zo=do*o*(1-o)
        g['Wfd_na']+=torch.outer(zf,conc); g['bfd_na']+=zf
        g['Wid_na']+=torch.outer(zi,conc); g['bid_na']+=zi
        g['Wcd_na']+=torch.outer(zc,conc); g['bcd_na']+=zc
        g['Wod_na']+=torch.outer(zo,conc); g['bod_na']+=zo
        dc2=Wfd_na.T@zf+Wid_na.T@zi+Wcd_na.T@zc+Wod_na.T@zo
        dh_dn=dc2[:d_hid]; dc_dn=dcc*f
        g['E_dec_na'][:, cc['emb_idx']] += dc2[d_hid:]

    dh_en = dh_dn.clone(); dc_en = dc_dn.clone()
    for t in reversed(range(T)):
        cc = ecs[t]
        # Sin atencion, h_t^enc solo recibe gradiente por la ruta recurrente:
        # todo tiene que viajar S + T pasos desde la perdida.
        f=cc['f']; i=cc['i']; ct=cc['ct']; c_t=cc['c_t']; o=cc['o']
        c_p=cc['c_prev']; conc=cc['concat']
        do=dh_en*torch.tanh(c_t); dcc=dc_en+dh_en*o*(1-torch.tanh(c_t)**2)
        df=dcc*c_p; di2=dcc*ct; dct2=dcc*i
        zf=df*f*(1-f); zi=di2*i*(1-i); zc=dct2*(1-ct**2); zo=do*o*(1-o)
        g['Wfe_na']+=torch.outer(zf,conc); g['bfe_na']+=zf
        g['Wie_na']+=torch.outer(zi,conc); g['bie_na']+=zi
        g['Wce_na']+=torch.outer(zc,conc); g['bce_na']+=zc
        g['Woe_na']+=torch.outer(zo,conc); g['boe_na']+=zo
        dc3=Wfe_na.T@zf+Wie_na.T@zi+Wce_na.T@zc+Woe_na.T@zo
        dh_en=dc3[:d_hid]; dc_en=dcc*f
        g['E_enc_na'][:, cc['emb_idx']] += dc3[d_hid:]

    return loss.item(), g


losses_no_attn = []
_params_na = ['E_enc_na','E_dec_na','Wfe_na','bfe_na','Wie_na','bie_na','Wce_na','bce_na',
              'Woe_na','boe_na','Wfd_na','bfd_na','Wid_na','bid_na','Wcd_na','bcd_na',
              'Wod_na','bod_na','Wout_na','bout_na']
for it in range(5):
    total = 0.0
    for src_s, tgt_s in TRAIN_DATA:
        l, g = fwd_bwd_sin_atencion(src_s, tgt_s)
        total += l
        for name in _params_na:
            _G[name] = _G[name] - alpha_lr * g[name]
    losses_no_attn.append(total / len(TRAIN_DATA))

print('Loss SIN atencion por iteracion:')
for i, l in enumerate(losses_no_attn): print(f'  Iter {i+1}: {l:.4f}')
print()
print(f'Reduccion con atencion: {(losses_train[0]-losses_train[-1])/losses_train[0]*100:.2f}%')
print(f'Reduccion sin atencion: {(losses_no_attn[0]-losses_no_attn[-1])/losses_no_attn[0]*100:.2f}%')

Loss SIN atencion por iteracion:
  Iter 1: 5.0730
  Iter 2: 5.0301
  Iter 3: 4.9875
  Iter 4: 4.9452
  Iter 5: 4.9032

Reduccion con atencion: 3.46%
Reduccion sin atencion: 3.35%


---
## Bloque 10: Visualizacion de pesos de atencion y comparacion de convergencia

In [26]:
import os
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)   # todas las figuras se guardan aqui

# ---- Comparacion de convergencia: con atencion vs sin atencion ----
if losses_train:
    plt.figure(figsize=(8,3.5))
    plt.plot(range(1,6), losses_train, 'o-', color='steelblue', lw=2, label='Con atencion')
    if losses_no_attn:
        plt.plot(range(1,6), losses_no_attn, 's--', color='indianred', lw=2,
                 label='Sin atencion (Semana 4)')
    plt.xlabel('Iteracion'); plt.ylabel('Loss promedio')
    plt.title('Convergencia Seq2Seq EN->ES: con vs sin atencion')
    plt.xticks(range(1,6))
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'convergencia_atencion.png'), dpi=110, bbox_inches='tight')
    plt.show()

# Visualizar mapa de atencion para un par de prueba
def get_attention_map(src_sentence, trained_params):
    """Calcula los pesos de atencion para cada paso del decoder."""
    Eenc,Edec,Wfe,bfe,Wie,bie,Wce,bce,Woe,boe,Wfd,bfd,Wid,bid,Wcd,bcd,Wod,bod,Wo2,bo2,WQ,WK,WV=trained_params
    src_t=tokenize_src(src_sentence)
    h=torch.zeros(d_hid); c=torch.zeros(d_hid); ec=[]
    for idx in src_t:
        emb=Eenc[:,idx]
        h,c,cache=lstm_cell(h,c,emb,Wfe,bfe,Wie,bie,Wce,bce,Woe,boe)
        cache['emb_idx']=idx; ec.append(cache)
    H=torch.stack([cc['h_t'] for cc in ec])
    K2=H@WK.T; V2=H@WV.T
    # Greedy decode recogiendo alphas
    h2=h.clone(); c2=c.clone()
    cur_idx=tgt_w2i[SOS]; alphas=[]; words_gen=[]
    for _ in range(10):
        emb=Edec[:,cur_idx]
        h2,c2,_=lstm_cell(h2,c2,emb,Wfd,bfd,Wid,bid,Wcd,bcd,Wod,bod)
        qs=WQ@h2; scs=K2@qs/(d_k**0.5); als=F.softmax(scs,dim=0)
        alphas.append(als.detach().numpy())
        cts=V2.T@als; h_cat=torch.cat([h2,cts]); z=Wo2@h_cat+bo2
        cur_idx=int(torch.argmax(z).item())
        word=tgt_i2w[cur_idx]
        if word==EOS: break
        words_gen.append(word)
    return alphas, words_gen, [src_i2w[i] for i in src_t]

# Usar parametros entrenados (si el bloque 9 esta completo)
if losses_train:
    params=(E_enc_tr,E_dec_tr,Wfe,bfe,Wie,bie,Wce,bce,Woe,boe,
            Wfd,bfd,Wid,bid,Wcd,bcd,Wod,bod,Wo2,bo2,WQ,WK,WV)
    src_ex='she sings well'
    alphas,words_gen,src_words=get_attention_map(src_ex,params)
    print(f'Entrada: {src_ex}')
    print(f'Generado: {" ".join(words_gen)}')
    if alphas:
        A=np.array(alphas)
        plt.figure(figsize=(6,4))
        plt.imshow(A,cmap='Blues',aspect='auto')
        plt.xticks(range(len(src_words)),src_words)
        plt.yticks(range(len(words_gen)),words_gen)
        plt.xlabel('Tokens encoder (EN)'); plt.ylabel('Pasos decoder (ES)')
        plt.title(f'Mapa de atencion: "{src_ex}"')
        plt.colorbar(label='Peso de atencion')
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, 'mapa_atencion.png'), dpi=110, bbox_inches='tight')
        plt.show()
        # Pesos numericos por paso: sirven para responder la Pregunta 2
        print()
        print('Pesos de atencion por paso del decoder:')
        print('paso'.ljust(12) + ''.join(w.rjust(10) for w in src_words))
        for w, row in zip(words_gen, A):
            print(w.ljust(12) + ''.join(f'{v:10.4f}' for v in row))
        print(f'\nEntropia media de alpha: '
              f'{float(-(A*np.log(A+1e-12)).sum(axis=1).mean()):.4f} nats '
              f'(uniforme = {float(np.log(len(src_words))):.4f})')

Entrada: she sings well
Generado: 

Pesos de atencion por paso del decoder:
paso               she     sings      well

Entropia media de alpha: 1.0986 nats (uniforme = 1.0986)


C:\Users\Admin\AppData\Local\Temp\ipykernel_31628\686628590.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Admin\AppData\Local\Temp\ipykernel_31628\686628590.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [27]:
# ---- Mapa de atencion con teacher forcing ----
# Con solo 5 iteraciones el decode greedy todavia colapsa en <EOS> en el primer paso,
# asi que el mapa greedy queda casi vacio. Alimentando el decoder con la traduccion de
# referencia se obtiene una fila de alphas por cada palabra generada, que es lo que
# necesita la Pregunta 2 para analizar la correspondencia EN->ES.
def mapa_atencion_teacher_forcing(src_sentence, tgt_sentence, trained_params):
    """Devuelve (alphas, palabras_objetivo, palabras_fuente) usando teacher forcing."""
    Eenc,Edec,Wfe,bfe,Wie,bie,Wce,bce,Woe,boe,Wfd,bfd,Wid,bid,Wcd,bcd,Wod,bod,Wo2,bo2,WQ,WK,WV=trained_params
    src_t = tokenize_src(src_sentence)
    h = torch.zeros(d_hid); c = torch.zeros(d_hid); ec = []
    for idx in src_t:
        h, c, cache = lstm_cell(h, c, Eenc[:, idx], Wfe,bfe,Wie,bie,Wce,bce,Woe,boe)
        ec.append(cache)
    H = torch.stack([cc['h_t'] for cc in ec])
    K2 = H @ WK.T; V2 = H @ WV.T

    tgt_ids = tokenize_tgt(tgt_sentence)
    h2 = h.clone(); c2 = c.clone(); alphas = []
    for in_idx in tgt_ids[:-1]:                      # <SOS> w1 w2 ... (teacher forcing)
        h2, c2, _ = lstm_cell(h2, c2, Edec[:, in_idx], Wfd,bfd,Wid,bid,Wcd,bcd,Wod,bod)
        als = F.softmax((K2 @ (WQ @ h2)) / (d_k ** 0.5), dim=0)
        alphas.append(als.detach().numpy())
    return alphas, [tgt_i2w[i] for i in tgt_ids[1:]], [src_i2w[i] for i in src_t]


if losses_train:
    src_ex = 'she sings well'; tgt_ex = 'canta bien'
    alphas_tf, tgt_words, src_words = mapa_atencion_teacher_forcing(src_ex, tgt_ex, params)
    A = np.array(alphas_tf)

    plt.figure(figsize=(6,4))
    plt.imshow(A, cmap='Blues', aspect='auto', vmin=0, vmax=A.max())
    plt.xticks(range(len(src_words)), src_words)
    plt.yticks(range(len(tgt_words)), tgt_words)
    plt.xlabel('Tokens encoder (EN)'); plt.ylabel('Pasos decoder (ES, teacher forcing)')
    plt.title(f'Mapa de atencion (teacher forcing): "{src_ex}" -> "{tgt_ex}"')
    plt.colorbar(label='Peso de atencion')
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            plt.text(j, i, f'{A[i,j]:.3f}', ha='center', va='center',
                     color='white' if A[i,j] > A.max()*0.6 else 'black', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'mapa_atencion_teacher_forcing.png'),
                dpi=110, bbox_inches='tight')
    plt.show()

    print('paso'.ljust(12) + ''.join(w.rjust(10) for w in src_words))
    for w, row in zip(tgt_words, A):
        print(w.ljust(12) + ''.join(f'{v:10.4f}' for v in row))
    _H_ent = float(-(A*np.log(A+1e-12)).sum(axis=1).mean())
    print(f'\nEntropia media: {_H_ent:.4f} nats  |  uniforme = {float(np.log(len(src_words))):.4f} nats')
    print('Cuanto menor la entropia, mas concentrado el mapa (ver Pregunta 2b).')

paso               she     sings      well
canta           0.3333    0.3333    0.3334
bien            0.3333    0.3333    0.3334
<EOS>           0.3333    0.3333    0.3334

Entropia media: 1.0986 nats  |  uniforme = 1.0986 nats
Cuanto menor la entropia, mas concentrado el mapa (ver Pregunta 2b).


C:\Users\Admin\AppData\Local\Temp\ipykernel_31628\2628110293.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Experimento adicional: 100 iteraciones (evidencia para la Pregunta 2b)

Con 5 iteraciones los pesos de atencion siguen practicamente uniformes ($\alpha \approx 1/T$,
entropia = $\log T$). Para comprobar empiricamente la prediccion de la Pregunta 2b se entrena
el mismo modelo 100 iteraciones y se mide, iteracion a iteracion, la **entropia** de la
distribucion de atencion: si el mapa se concentra, la entropia debe bajar desde $\log T$.

In [28]:
# Snapshot de los pesos de 5 iteraciones para no perder el estado de los bloques anteriores
params_5it = {n: _G[n].clone() for n in _params_attn}

# Reinicializacion identica al Bloque 9
torch.manual_seed(42)
E_enc_tr=torch.randn(d_emb,src_V)*0.1; E_dec_tr=torch.randn(d_emb,tgt_V)*0.1
Wfe=torch.randn(d_hid,d_hid+d_emb)*0.1; bfe=torch.zeros(d_hid)
Wie=torch.randn(d_hid,d_hid+d_emb)*0.1; bie=torch.zeros(d_hid)
Wce=torch.randn(d_hid,d_hid+d_emb)*0.1; bce=torch.zeros(d_hid)
Woe=torch.randn(d_hid,d_hid+d_emb)*0.1; boe=torch.zeros(d_hid)
Wfd=torch.randn(d_hid,d_hid+d_emb)*0.1; bfd=torch.zeros(d_hid)
Wid=torch.randn(d_hid,d_hid+d_emb)*0.1; bid=torch.zeros(d_hid)
Wcd=torch.randn(d_hid,d_hid+d_emb)*0.1; bcd=torch.zeros(d_hid)
Wod=torch.randn(d_hid,d_hid+d_emb)*0.1; bod=torch.zeros(d_hid)
Wo2=torch.randn(tgt_V,d_hid+d_v)*0.1; bo2=torch.zeros(tgt_V)
torch.manual_seed(7)
WQ=torch.randn(d_k,d_hid)*0.1; WK=torch.randn(d_k,d_hid)*0.1; WV=torch.randn(d_v,d_hid)*0.1

N_ITERS = 100
src_ex, tgt_ex = 'she sings well', 'canta bien'
losses_100 = []; entropias = []
for it in range(N_ITERS):
    total = 0.0
    for src_s, tgt_s in TRAIN_DATA:
        l, g = fwd_bwd_atencion(src_s, tgt_s)
        total += l
        for name in _params_attn:
            _G[name] = _G[name] - alpha_lr * g[name]
    losses_100.append(total / len(TRAIN_DATA))
    # Entropia media del mapa de atencion en el ejemplo de la Pregunta 2
    A_it, _, src_words = mapa_atencion_teacher_forcing(
        src_ex, tgt_ex, tuple(_G[n] for n in _params_attn))
    A_it = np.array(A_it)
    entropias.append(float(-(A_it*np.log(A_it+1e-12)).sum(axis=1).mean()))

H_unif = float(np.log(len(src_words)))
print(f'Loss: {losses_100[0]:.4f} (iter 1) -> {losses_100[-1]:.4f} (iter {N_ITERS})  '
      f'[{(losses_100[0]-losses_100[-1])/losses_100[0]*100:.2f}% reduccion]')
print(f'Entropia de alpha: {entropias[0]:.4f} -> {entropias[-1]:.4f} nats '
      f'(uniforme = {H_unif:.4f})')

fig, ax = plt.subplots(1, 2, figsize=(11,3.5))
ax[0].plot(range(1,N_ITERS+1), losses_100, color='steelblue', lw=2)
ax[0].set_xlabel('Iteracion'); ax[0].set_ylabel('Loss promedio')
ax[0].set_title(f'Convergencia con atencion ({N_ITERS} iteraciones)')
ax[0].grid(True, alpha=0.3)
ax[1].plot(range(1,N_ITERS+1), entropias, color='darkorange', lw=2, label='Entropia de alpha')
ax[1].axhline(H_unif, ls='--', color='gray', label=f'Uniforme (log T = {H_unif:.3f})')
ax[1].set_xlabel('Iteracion'); ax[1].set_ylabel('Entropia media (nats)')
ax[1].set_title(f'Concentracion del mapa: "{src_ex}"')
ax[1].legend(); ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'convergencia_100_iteraciones.png'), dpi=110, bbox_inches='tight')
plt.show()

# Mapa de atencion tras 100 iteraciones
A100, tgt_words, src_words = mapa_atencion_teacher_forcing(
    src_ex, tgt_ex, tuple(_G[n] for n in _params_attn))
A100 = np.array(A100)
plt.figure(figsize=(6,4))
plt.imshow(A100, cmap='Blues', aspect='auto')
plt.xticks(range(len(src_words)), src_words)
plt.yticks(range(len(tgt_words)), tgt_words)
plt.xlabel('Tokens encoder (EN)'); plt.ylabel('Pasos decoder (ES)')
plt.title(f'Mapa de atencion tras {N_ITERS} iteraciones')
plt.colorbar(label='Peso de atencion')
for i in range(A100.shape[0]):
    for j in range(A100.shape[1]):
        plt.text(j, i, f'{A100[i,j]:.3f}', ha='center', va='center',
                 color='white' if A100[i,j] > A100.max()*0.6 else 'black', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'mapa_atencion_100_iteraciones.png'), dpi=110, bbox_inches='tight')
plt.show()

print('paso'.ljust(12) + ''.join(w.rjust(10) for w in src_words))
for w, row in zip(tgt_words, A100):
    print(w.ljust(12) + ''.join(f'{v:10.4f}' for v in row))

# Restaurar los pesos de 5 iteraciones (estado que evaluan los bloques anteriores)
for n in _params_attn: _G[n] = params_5it[n]
print('\nPesos de 5 iteraciones restaurados.')

Loss: 5.0725 (iter 1) -> 3.7497 (iter 100)  [26.08% reduccion]
Entropia de alpha: 1.0986 -> 1.0986 nats (uniforme = 1.0986)


paso               she     sings      well
canta           0.3332    0.3333    0.3334
bien            0.3332    0.3334    0.3335
<EOS>           0.3331    0.3334    0.3335

Pesos de 5 iteraciones restaurados.


C:\Users\Admin\AppData\Local\Temp\ipykernel_31628\1602942127.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Admin\AppData\Local\Temp\ipykernel_31628\1602942127.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura del experimento (evidencia empirica para la Pregunta 2b).**

El resultado es contraintuitivo y vale la pena documentarlo: la loss **si** baja (5.0725 -> 3.7497, 26% en 100 iteraciones),
pero la entropia de $\boldsymbol{\alpha}$ se queda clavada en $\log 3 = 1.0986$ nats, es decir el mapa sigue **uniforme**.
La causa se ve en las magnitudes del Bloque 2:

- Los hidden states del encoder son diminutos (`H_enc.abs().mean()` $\approx 8.5\times10^{-3}$), porque tanto los embeddings
  como los pesos del LSTM se inicializan con escala $0.1$.
- Por eso los scores son $e_{0,t} \approx 1$-$3 \times 10^{-5}$, y $\text{softmax}$ de valores casi identicos da
  $\alpha_t \approx 1/T$ exactamente.
- Con $\alpha$ uniforme, el jacobiano del softmax $\text{diag}(\alpha) - \alpha\alpha^\top$ es casi singular en la direccion
  constante: **anula** casi todo el gradiente que iba hacia los scores. Numericamente,
  $\|\partial L/\partial W_V\|_\infty \approx 2\times10^{-3}$ frente a $\|\partial L/\partial W_Q\|_\infty \approx 3.5\times10^{-8}$:
  cinco ordenes de magnitud de diferencia.

Es decir: **la Ruta 1 (values) aprende y la Ruta 2 (scores) practicamente no**, asi que el modelo mejora usando la atencion
como una media fija de los values, no como un selector. Es un punto fijo pobre pero no inestable: la prediccion teorica de la
Pregunta 2b (el mapa se concentra al entrenar mas) es correcta *en el regimen adecuado*, y para llegar a el hace falta romper
la escala del score: inicializacion mas grande (p. ej. $\mathcal{N}(0,1/\sqrt{d})$ en lugar de $0.1\cdot\mathcal{N}(0,1)$),
un learning rate mayor para $W_Q$/$W_K$, o normalizacion de los hidden states antes de proyectarlos. Con esta configuracion
del laboratorio, 100 iteraciones no bastan.

---
## Bloque 11: Preguntas de analisis

---

### Pregunta 1 (35 pts)

En el Bloque 6, el gradiente de la perdida respecto a los pesos de atencion $\boldsymbol{\alpha}_s$ fluye en dos rutas distintas desde $\tilde{\mathbf{c}}_s$, y desde ahi hacia $W_Q$, $W_K$ y $W_V$.

a) Trace las dos rutas del backward desde $\tilde{\mathbf{c}}_s$ hasta $H_{enc}$, explicando matematicamente que calcula cada ruta y por que son necesarias las dos. ¿Que informacion aprende el modelo a traves de cada ruta que no podria aprender sin ella?

b) En el seq2seq sin atencion, el gradiente viaja $S + T$ pasos desde la perdida hasta los pesos del encoder. Con atencion, el gradiente hacia $\mathbf{h}_t^{enc}$ llega escalado por $\alpha_{s,t}$. Explique en terminos del grafo de computo como cambia el recorrido del gradiente y que implicacion tiene el factor $\alpha_{s,t}$ para el aprendizaje de tokens que raramente reciben atencion.

c) Las matrices $W_Q$, $W_K$ y $W_V$ reciben el mismo hidden state del encoder como entrada pero aprenden proyecciones distintas. Sin que nadie se lo programe, ¿que diferencia esperaria observar en lo que aprende $W_Q$ versus lo que aprende $W_K$, dado que $W_Q$ proyecta el decoder y $W_K$ proyecta el encoder? Justifique en terminos del gradiente que recibe cada una.

**Su respuesta a la Pregunta 1:**

a) Básicamente el backward desde el vector de contexto ($\tilde{\mathbf{c}}_s = \sum_t \alpha_{s,t} \mathbf{v}_t$) se divide en dos caminos. La **Ruta 1** va directo hacia la matriz de values ($V_{mat}$) y $W_V$. Matemáticamente, el gradiente fluye como $\frac{\partial \tilde{\mathbf{c}}_s}{\partial \mathbf{v}_t} = \alpha_{s,t}$. Esto demuestra por qué el gradiente que pasa por aquí es directamente proporcional a ese $\alpha$. Esta ruta sirve para que el modelo aprenda "qué" información (el contenido puro) debe extraer del encoder para mandarla al decoder.

Por otro lado, la **Ruta 2** retrocede desde $\tilde{\mathbf{c}}_s$ hacia los pesos $\alpha$. Matemáticamente, el gradiente inicial fluye como $\frac{\partial \tilde{\mathbf{c}}_s}{\partial \alpha_{s,t}} = \mathbf{v}_t$. Luego, aplicando la regla de la cadena, este gradiente atraviesa la función Softmax hacia los scores ($e_{s,t} = \mathbf{q}_s \cdot \mathbf{k}_t$) para finalmente actualizar los pesos $W_Q$ y $W_K$. Esta ruta le enseña a $W_Q$ y $W_K$ a "hacer match"; es decir, aprende a decidir a qué tokens darles importancia. Son necesarias las dos porque sin la ruta 1 el modelo no sabría qué información pasar, y sin la ruta 2 no sabría a dónde mirar.


b) En el seq2seq de la semana pasada, el gradiente tenía que hacer todo el recorrido largo paso por paso (desenrollando el LSTM S+T veces), lo que hacía que el gradiente se fuera perdiendo en el camino (el famoso vanishing gradient). Con la atención, el grafo de cómputo crea literalmente atajos directos entre el paso actual del decoder y cualquier paso del encoder. 

El rollo con que llegue escalado por $\alpha_{s,t}$ es que este peso funciona como una válvula. Si el modelo casi nunca le presta atención a un token (su $\alpha$ es cercano a 0), la válvula se cierra y casi no le llega gradiente. Esto significa que el modelo no va a actualizar mucho los pesos asociados a ese token irrelevante, dejando que los gradientes fuertes se enfoquen en los tokens que de verdad importan para la traducción.


c) Aunque al principio pareciera que hacen lo mismo, el gradiente las especializa por su ubicación. $W_Q$ está del lado del decoder, así que su gradiente viene de la necesidad de predecir la siguiente palabra; por lo tanto, aprende a comportarse como un buen "buscador" que resume qué es lo que le falta a la oración (por ejemplo, "necesito un verbo ahora"). 

En cambio, $W_K$ procesa los estados del encoder, así que su gradiente la obliga a funcionar como un buen "índice" o catálogo, creando etiquetas para que cuando $W_Q$ busque algo, $W_K$ pueda responder "aquí está el sujeto" o "aquí está el verbo". Nadie se los programó así, pero el simple flujo del gradiente fuerza esta dinámica de buscador y catálogo.

---
### Pregunta 2 (35 pts)

Observe el mapa de atencion generado en el Bloque 10 para la oracion 'she sings well' -> 'canta bien'.

a) La oracion en ingles tiene sujeto ('she') y la traduccion al espanol lo elide ('canta bien' sin 'ella'). En el paso del decoder que genera 'canta', ¿que tokens del encoder esperaria que recibieran mayor peso de atencion y por que? ¿Es el sujeto 'she' relevante para generar 'canta'? Conecte su respuesta con lo que el mecanismo de atencion esta matematicamente calculando.

b) Si entrenara el mismo modelo durante 100 iteraciones en lugar de 5, ¿esperaria que el mapa de atencion se volviera mas o menos concentrado (pesos mas o menos uniformes)? Justifique en terminos de lo que el modelo aprende progresivamente sobre las correspondencias entre palabras en ingles y espanol.

c) Proponga un caso especifico del corpus donde el mecanismo de atencion de **producto punto escalado** tendria dificultad para capturar la correspondencia correcta, y explique por que. ¿Que alternativa arquitectonica (sin necesidad de implementarla) resolveria ese caso? Justifique matematicamente.

**Su respuesta a la Pregunta 2:**

a) Para generar **"canta"**, esperaría que el mayor peso se concentrara en **"sings"**, porque es el token que aporta directamente el significado del verbo. Sin embargo, **"she"** también es relevante: aunque no se traduzca como una palabra separada, indica tercera persona singular y ayuda a escoger la forma conjugada "canta". Matemáticamente, el decoder calcula los scores $e_{s,t}=q_s^T k_t/\sqrt{d_k}$; si el query del paso de "canta" es compatible con las keys de "sings" y "she", sus scores y, después del softmax, sus pesos $\alpha_{s,t}$ serán mayores. El contexto $\tilde{c}_s=\sum_t \alpha_{s,t}v_t$ combina entonces la información semántica de "sings" con la información gramatical de "she".

b) En general esperaría un mapa **más concentrado** después de 100 iteraciones. Al inicio las proyecciones $W_Q$ y $W_K$ son casi aleatorias, por lo que los productos punto suelen ser parecidos y el softmax produce una distribución relativamente uniforme. Con más entrenamiento, el modelo puede aumentar la compatibilidad entre el query de cada palabra en español y las keys de los tokens ingleses que mejor la explican; por ejemplo, "canta" con "sings". Esto separa los scores y hace que el softmax asigne más probabilidad a unas pocas posiciones. No necesariamente se concentraría en un solo token, ya que la conjugación también puede requerir información del sujeto.

c) Un caso difícil sería **"she visits her friend" → "visita a su amiga"**, especialmente al generar **"su"**. La palabra española depende tanto de "her" como del sujeto "she", y además el género de "amiga" debe mantenerse coherente. Una sola atención de producto punto genera un score escalar por token y resume todo en un único vector; por eso puede costarle representar al mismo tiempo la relación posesiva, el sujeto y el género. Una alternativa sería **multi-head attention**, donde cada cabeza usa proyecciones distintas: $\text{head}_i=\text{softmax}(QW_Q^{(i)}(KW_K^{(i)})^T/\sqrt{d_k})VW_V^{(i)}$. Así, una cabeza podría atender a "her", otra a "she" y otra a "friend", conservando varias relaciones antes de combinar sus resultados.

---
### Pregunta 3 (30 pts)

Esta pregunta explora self-attention conceptualmente, como preparacion para la Semana 6.

a) En el mecanismo de atencion que implemento, los queries vienen del decoder y las keys y values del encoder. En self-attention, los tres vienen de la misma secuencia. Identifique exactamente que lineas de codigo del Bloque 6 cambiarian si convirtiera su implementacion de cross-attention a self-attention sobre la secuencia del encoder. ¿Que nueva informacion capturaria el modelo que la arquitectura LSTM no puede capturar directamente?

b) En self-attention, cada posicion calcula su query usando $W_Q$ y cada posicion ofrece su key usando $W_K$. Ambas matrices se inicializan aleatoriamente e identicamente podrian converger al mismo valor. Sin embargo, en la practica aprenden proyecciones distintas. Explique matematicamente, usando la estructura del grafo de computo, por que el gradiente que llega a $W_Q$ es diferente al que llega a $W_K$, incluso cuando reciben los mismos vectores de entrada.

c) Un Transformer con 8 cabezas de atencion ('multi-head attention') aplica 8 mecanismos de atencion en paralelo sobre la misma secuencia. Si todas las cabezas compartieran las mismas matrices $W_Q$, $W_K$, $W_V$, ¿que pasaria con el gradiente durante el entrenamiento y por que ese diseno no funcionaria bien? Justifique en terminos de la diversidad de representaciones que el modelo necesita aprender.

**Su respuesta a la Pregunta 3:**

a) **Lineas del Bloque 6 que cambian al pasar de cross-attention a self-attention sobre la secuencia del encoder.**

En el codigo actual el query sale del decoder y las keys/values del encoder. En self-attention sobre $H_{enc}$ los tres salen de la misma matriz $H_{enc}$, es decir $Q = H_{enc}W_Q^\top$, $K = H_{enc}W_K^\top$, $V = H_{enc}W_V^\top$, los scores dejan de ser un vector $(T,)$ y pasan a ser una matriz $E = QK^\top/\sqrt{d_k} \in \mathbb{R}^{T\times T}$ con softmax por filas. Concretamente:

| Linea actual (cross-attention) | Cambio en self-attention |
|---|---|
| `q_s = W_Q @ h2` (Bloque 5) | `Q = H_enc @ W_Q.T` → una query por token del encoder, no una por paso del decoder |
| `dW_Q += torch.outer(dq_s, ac['h_dec'])` | `dW_Q += dQ.T @ H_enc` → el gradiente de $W_Q$ se contrae contra $H_{enc}$, no contra $h_s^{dec}$ |
| `dh_s = dh_s + W_Q.T @ dq_s` | **desaparece**; se reemplaza por `dH_enc += dQ @ W_Q`, porque el query ya no viene del decoder sino del propio encoder |
| `dsc = alpha * (dal - dot(dal, alpha))` | se aplica **fila por fila** de la matriz $A \in \mathbb{R}^{T\times T}$: `dE = A * (dA - (dA*A).sum(dim=1, keepdim=True))` |
| `dH_enc` recibe 2 contribuciones (`dK_s @ W_K`, `dV_s @ W_V`) | recibe **3**: la de keys, la de values y la nueva de queries |

Es decir: cambia el origen del query (una linea en el forward), la forma de los tensores de scores/alphas (vector → matriz, softmax por filas) y, en el backward, la tercera ruta de gradiente que ahora tambien cae sobre $H_{enc}$.

**Que informacion nueva captura.** El LSTM comprime el contexto de forma secuencial y unidireccional: $h_t^{enc}$ solo ve $x_1..x_t$ y a traves de un unico cuello de botella de $d_{hid}=32$ dimensiones, con un camino de $|i-j|$ pasos entre dos tokens. Con self-attention cada token compara su query contra **todas** las keys de la secuencia (incluidas las de la derecha) en un solo paso: el camino entre dos posiciones cualesquiera pasa a ser $O(1)$. Eso permite capturar directamente relaciones token-token que el LSTM solo puede aproximar: concordancia sujeto-verbo a distancia ('the teacher ... explains'), correferencia, desambiguacion por contexto posterior, y en general un enrutamiento de informacion basado en **contenido** ($q^\top k$) y no en **posicion en el tiempo**. Ademas los $T$ calculos son independientes entre si, asi que se paralelizan, mientras que el LSTM es inherentemente secuencial.

b) **Por que $W_Q$ y $W_K$ reciben gradientes distintos aunque vean los mismos vectores.**

El score es una **forma bilineal**: $e_{ij} = \dfrac{(W_Q h_i)^\top (W_K h_j)}{\sqrt{d_k}} = \dfrac{h_i^\top W_Q^\top W_K h_j}{\sqrt{d_k}}$. Derivando:

$$\frac{\partial L}{\partial W_Q} = \frac{1}{\sqrt{d_k}}\sum_{i,j} \frac{\partial L}{\partial e_{ij}} \, (W_K h_j)\, h_i^\top, \qquad \frac{\partial L}{\partial W_K} = \frac{1}{\sqrt{d_k}}\sum_{i,j} \frac{\partial L}{\partial e_{ij}} \, (W_Q h_i)\, h_j^\top$$

En el codigo esto son exactamente las lineas `dq_s = K_mat.T @ dscr` (el gradiente del query se contrae contra las **keys**) y `dK_s = torch.outer(dscr, ac['q'])` (el gradiente de las keys se contrae contra el **query**). Cada matriz recibe como factor izquierdo la proyeccion de *la otra* y como factor derecho *su propia* entrada. Son operaciones distintas incluso con $h_i = h_j$.

Supongamos el peor caso: $W_Q = W_K = W$ e identica entrada. Entonces $\partial L/\partial W_Q = \frac{1}{\sqrt{d_k}}\sum_{ij} g_{ij}(Wh_j)h_i^\top$ y $\partial L/\partial W_K = \frac{1}{\sqrt{d_k}}\sum_{ij} g_{ij}(Wh_i)h_j^\top$, con $g_{ij} = \partial L/\partial e_{ij}$. Ambas expresiones coinciden solo si $g$ es **simetrica** ($g_{ij} = g_{ji}$). No lo es: el softmax se aplica **por filas**, asi que $g_{ij} = \alpha_{ij}(dA_{ij} - \sum_k dA_{ik}\alpha_{ik})$ normaliza sobre $j$ pero no sobre $i$, y ademas la fila $i$ pondera la perdida del token $i$ y no la del $j$. Esa asimetria estructural del grafo de computo rompe la simetria desde el primer paso de gradiente y las dos matrices divergen. Semanticamente, $W_Q$ aprende a codificar *"que estoy buscando"* y $W_K$ *"que ofrezco"*: son los dos lados de una consulta tipo clave-valor, y por eso $W_Q^\top W_K$ termina siendo una matriz de compatibilidad **no simetrica** ("adjetivo busca sustantivo" no implica "sustantivo busca adjetivo"). En el cross-attention del laboratorio la asimetria es todavia mas evidente: $W_Q$ solo ve estados del decoder y $W_K$ solo estados del encoder, asi que ni siquiera reciben las mismas entradas.

c) **Multi-head con $W_Q$, $W_K$, $W_V$ compartidas entre las 8 cabezas.**

Si las 8 cabezas comparten exactamente las mismas matrices y reciben la misma secuencia de entrada, calculan la **misma funcion determinista**: mismos scores, mismos $\alpha$, mismo contexto. Las 8 salidas serian 8 copias identicas de un solo head, y la concatenacion $[\text{head}_1;...;\text{head}_8]$ seria el mismo vector repetido 8 veces: rango efectivo 1, no 8. El modelo tendria la capacidad de una sola cabeza pero pagando el costo de computo de ocho.

En el gradiente el problema es peor, porque es un caso clasico de **simetria no rota**: como $W$ es un unico parametro compartido,

$$\frac{\partial L}{\partial W_Q} = \sum_{m=1}^{8} \frac{\partial L}{\partial W_Q^{(m)}} = 8 \cdot \frac{\partial L}{\partial W_Q^{(1)}}$$

(los 8 terminos son identicos porque las cabezas lo son). El update es el mismo gradiente escalado por 8, o sea equivale a entrenar **una** cabeza con learning rate $8\times$ mayor: mas riesgo de inestabilidad y explosion, cero diversidad ganada. Y como el update es identico para todas, las cabezas que empezaron iguales **siguen iguales para siempre**: no existe ninguna fuente de asimetria que las separe (el mismo argumento por el que no se puede inicializar una capa densa con todos los pesos iguales).

Lo que hace funcionar al multi-head es justamente lo contrario: cada cabeza $m$ tiene sus propias $W_Q^{(m)}, W_K^{(m)}, W_V^{(m)}$ inicializadas de forma independiente, proyecta a un subespacio distinto de dimension $d_k = d_{model}/h$, y recibe un gradiente distinto (via su columna correspondiente de $W_O$). Asi cada cabeza se especializa en un patron de relacion diferente —una en concordancia sujeto-verbo, otra en dependencias posicionales locales, otra en correferencia, otra en el token anterior— y la concatenacion entrega al modelo un conjunto **rico y complementario** de relaciones. La unica variante compartida que si funciona es compartir solo $K$ y $V$ entre cabezas manteniendo $W_Q$ separada por cabeza (*multi-query / grouped-query attention*), precisamente porque el query distinto por cabeza basta para romper la simetria y hacer que cada cabeza atienda a algo distinto.

---
## Bloque 12: Nota automatica sobre la seccion de codigo

In [29]:
_PUNTOS = {
    'b1': ('Bloque 1: Proyecciones Q, K, V',        8),
    'b2': ('Bloque 2: Attention scores',             8),
    'b3': ('Bloque 3: Attention weights',            6),
    'b4': ('Bloque 4: Vector de contexto dinamico',  6),
    'b5': ('Bloque 5: Forward decoder con atencion', 14),
    'b6': ('Bloque 6: Backward de atencion',         10),
    'b8': ('Bloque 8: Actualizacion de parametros',  5),
    'b9': ('Bloque 9: Convergencia 5 iteraciones',   3),
}
_TOTAL = 60
print('='*62)
print('  NOTA AUTOMATICA - SECCION DE CODIGO')
print('='*62)
_obtenido=0
for key,(nombre,pts_max) in _PUNTOS.items():
    val=_resultados.get(key,False)
    pts=pts_max if val is True else 0
    _obtenido+=pts
    print(f'  {"CORRECTO" if val is True else "PENDIENTE":10s} | {nombre:38s} | {pts:2d}/{pts_max} pts')
print('-'*62)
print(f'  Subtotal codigo:   {_obtenido}/{_TOTAL} puntos')
print('  Pendiente manual:')
print('    Bloque 11 preguntas : __/25 pts')
print('    Comentarios codigo  : __/15 pts')
print('-'*62)
print('  TOTAL FINAL (sobre 100): __/100 pts')
print('='*62)

  NOTA AUTOMATICA - SECCION DE CODIGO
  CORRECTO   | Bloque 1: Proyecciones Q, K, V         |  8/8 pts
  CORRECTO   | Bloque 2: Attention scores             |  8/8 pts
  CORRECTO   | Bloque 3: Attention weights            |  6/6 pts
  CORRECTO   | Bloque 4: Vector de contexto dinamico  |  6/6 pts
  CORRECTO   | Bloque 5: Forward decoder con atencion | 14/14 pts
  CORRECTO   | Bloque 6: Backward de atencion         | 10/10 pts
  PENDIENTE  | Bloque 8: Actualizacion de parametros  |  0/5 pts
  CORRECTO   | Bloque 9: Convergencia 5 iteraciones   |  3/3 pts
--------------------------------------------------------------
  Subtotal codigo:   55/60 puntos
  Pendiente manual:
    Bloque 11 preguntas : __/25 pts
    Comentarios codigo  : __/15 pts
--------------------------------------------------------------
  TOTAL FINAL (sobre 100): __/100 pts
